<a href="https://colab.research.google.com/github/japmanyakaur/early-disease-detection-in-crops/blob/main/crop_disease_TRAINING_MOREDATA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Crop Disease Detection — v10 MORE DATA (Tomato + Grape)

close the lab→field accuracy gap by adding *real, class-relevant*
field photo datasets on top of PlantVillage (lab) + PlantDoc (field), instead of chasing
more training tricks on the same limited field data.

- **Grape expands from 4 -> 7 classes.** Added the Niphad Grape Leaf Disease Dataset (NGLD):
  2,726 real vineyard photos from Nashik, Maharashtra (2023-2025), covering Downy Mildew,
  
- **Tomato stays at 10 classes**
  (Early Blight, Late Blight, Leaf Mold, Bacterial Spot, Target Spot) from a 731-image
  Bangladesh field dataset (Dinajpur/Thakurgaon/Kushtia). Its "Black Spot" class

## Section 1 — Mount Drive & project folders

One dedicated folder for this version: `crop-disease-model/v10_moredata/`.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os

BASE = '/content/drive/MyDrive/crop-disease-model'
PROJECT_DIR = f'{BASE}/v10_moredata'
CKPT_DIR = f'{PROJECT_DIR}/checkpoints'
MODELS_DIR = f'{PROJECT_DIR}/final_models'
DATA_CACHE_DIR = f'{PROJECT_DIR}/data_cache'
MANUAL_DOWNLOADS_DIR = f'{PROJECT_DIR}/manual_downloads'

for d in [PROJECT_DIR, CKPT_DIR, MODELS_DIR, DATA_CACHE_DIR, MANUAL_DOWNLOADS_DIR]:
    os.makedirs(d, exist_ok=True)

print('Project folder for this version:')
print(PROJECT_DIR)
print('\nEverything this notebook saves (checkpoints, final models, cached datasets)')
print('lives under this one folder. v9 and earlier versions are untouched.')


Mounted at /content/drive
Project folder for this version:
/content/drive/MyDrive/crop-disease-model/v10_moredata

Everything this notebook saves (checkpoints, final models, cached datasets)
lives under this one folder. v9 and earlier versions are untouched.


## Section 2 — Install dependencies

In [2]:
!pip install -q timm kagglehub pytorch-grad-cam
print('done')


ERROR: Could not find a version that satisfies the requirement pytorch-grad-cam (from versions: none)
ERROR: No matching distribution found for pytorch-grad-cam
done


## Section 3 — Download PlantVillage (lab photos) and filter to Tomato + Grape

Same source as every earlier version. Cached to Drive after first download so a
disconnect never re-downloads it.

In [ ]:
import kagglehub, shutil, glob

PV_CACHE = f'{DATA_CACHE_DIR}/plantvillage_raw'

if not os.path.exists(PV_CACHE):
    pv_path = kagglehub.dataset_download('vipoooool/new-plant-diseases-dataset')
    shutil.copytree(pv_path, PV_CACHE)
    print('Downloaded and cached PlantVillage to Drive.')
else:
    print('PlantVillage already cached in Drive, skipping download.')

# Locate the actual train/valid folders (kagglehub nests these differently across versions)
def find_dir(root, name_contains):
    for dirpath, dirnames, _ in os.walk(root):
        for d in dirnames:
            if name_contains.lower() in d.lower():
                return os.path.join(dirpath, d)
    return None

PV_TRAIN = find_dir(PV_CACHE, 'train')
PV_VALID = find_dir(PV_CACHE, 'valid')
assert PV_TRAIN and PV_VALID, 'Could not locate PlantVillage train/valid folders — inspect PV_CACHE manually.'
print('PlantVillage train dir:', PV_TRAIN)
print('PlantVillage valid dir:', PV_VALID)


Using Colab cache for faster access to the 'new-plant-diseases-dataset' dataset.


In [ ]:
# Canonical class lists (folder-name -> clean class name)
TOMATO_CLASSES = [
    'Tomato___Bacterial_spot',
    'Tomato___Early_blight',
    'Tomato___Late_blight',
    'Tomato___Leaf_Mold',
    'Tomato___Septoria_leaf_spot',
    'Tomato___Spider_mites Two-spotted_spider_mite',
    'Tomato___Target_Spot',
    'Tomato___Tomato_Yellow_Leaf_Curl_Virus',
    'Tomato___Tomato_mosaic_virus',
    'Tomato___healthy',
]

# Grape now has 7 classes: the original PlantVillage 3 diseases + healthy,
# plus 3 new disease classes contributed entirely by NGLD (PlantVillage/PlantDoc
# have zero images for these three).
GRAPE_CLASSES = [
    'Grape___Black_rot',
    'Grape___Esca_(Black_Measles)',
    'Grape___Leaf_blight_(Isariopsis_Leaf_Spot)',
    'Grape___Downy_Mildew',
    'Grape___Powdery_Mildew',
    'Grape___Bacterial_Rot',
    'Grape___healthy',
]

CROP_CLASSES = {'tomato': TOMATO_CLASSES, 'grape': GRAPE_CLASSES}

def build_lab_split(pv_dir, classes, out_dir):
    """Copy only the folders we care about out of the huge PlantVillage tree."""
    os.makedirs(out_dir, exist_ok=True)
    for cls in classes:
        src = os.path.join(pv_dir, cls)
        dst = os.path.join(out_dir, cls)
        if os.path.exists(src) and not os.path.exists(dst):
            shutil.copytree(src, dst)

for crop, classes in CROP_CLASSES.items():
    build_lab_split(PV_TRAIN, classes, f'{DATA_CACHE_DIR}/{crop}/lab_train')
    build_lab_split(PV_VALID, classes, f'{DATA_CACHE_DIR}/{crop}/lab_val')

print('PlantVillage filtered into per-crop lab_train / lab_val folders.')
for crop in CROP_CLASSES:
    n_train = sum(len(glob.glob(f'{DATA_CACHE_DIR}/{crop}/lab_train/*/*')))
    n_val = sum(len(glob.glob(f'{DATA_CACHE_DIR}/{crop}/lab_val/*/*')))
    print(f'{crop}: {n_train} lab_train images, {n_val} lab_val images')


## Section 4 — Download PlantDoc (field photos) and split by crop

Uses PlantDoc's own train/test split: `train/` becomes field_finetune data,
`test/`

In [ ]:
PLANTDOC_DIR = f'{DATA_CACHE_DIR}/plantdoc_raw'

if not os.path.exists(PLANTDOC_DIR):
    !git clone --depth 1 https://github.com/pratikkayal/PlantDoc-Dataset.git {PLANTDOC_DIR}
else:
    print('PlantDoc already cached, skipping clone.')

import os as _os
print('PlantDoc top-level contents:', _os.listdir(PLANTDOC_DIR))


In [ ]:
# Map PlantDoc's own (messier) folder names to our canonical class names.
PLANTDOC_NAME_MAP = {
    'Tomato Early blight leaf': 'Tomato___Early_blight',
    'Tomato Septoria leaf spot': 'Tomato___Septoria_leaf_spot',
    'Tomato leaf bacterial spot': 'Tomato___Bacterial_spot',
    'Tomato leaf late blight': 'Tomato___Late_blight',
    'Tomato leaf mosaic virus': 'Tomato___Tomato_mosaic_virus',
    'Tomato leaf yellow virus': 'Tomato___Tomato_Yellow_Leaf_Curl_Virus',
    'Tomato mold leaf': 'Tomato___Leaf_Mold',
    'Tomato two spotted spider mites leaf': 'Tomato___Spider_mites Two-spotted_spider_mite',
    'Tomato leaf': 'Tomato___healthy',
    'grape leaf black rot': 'Grape___Black_rot',
    'Grape leaf black rot': 'Grape___Black_rot',
    'grape leaf': 'Grape___healthy',
    'Grape leaf': 'Grape___healthy',
}

def build_field_split(plantdoc_split_dir, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    if not os.path.isdir(plantdoc_split_dir):
        return
    for folder_name in os.listdir(plantdoc_split_dir):
        canon = PLANTDOC_NAME_MAP.get(folder_name)
        if canon is None:
            continue
        src = os.path.join(plantdoc_split_dir, folder_name)
        dst = os.path.join(out_dir, canon)
        os.makedirs(dst, exist_ok=True)
        for fname in os.listdir(src):
            dst_path = os.path.join(dst, fname)
            if not os.path.exists(dst_path):
                shutil.copy2(os.path.join(src, fname), dst_path)

for crop in CROP_CLASSES:
    build_field_split(f'{PLANTDOC_DIR}/train', f'{DATA_CACHE_DIR}/{crop}/field_finetune')
    build_field_split(f'{PLANTDOC_DIR}/test', f'{DATA_CACHE_DIR}/{crop}/field_eval')

print('PlantDoc split into per-crop field_finetune / field_eval folders.')
for crop in CROP_CLASSES:
    n_ft = sum(len(glob.glob(f'{DATA_CACHE_DIR}/{crop}/field_finetune/*/*')))
    n_ev = sum(len(glob.glob(f'{DATA_CACHE_DIR}/{crop}/field_eval/*/*')))
    print(f'{crop}: {n_ft} field_finetune images (from PlantDoc train), {n_ev} field_eval images (from PlantDoc test)')


## Section 5 — NGLD (Niphad Grape Leaf Disease Dataset) — grape expansion



In [ ]:
NGLD_ZIP = f'{MANUAL_DOWNLOADS_DIR}/NGLD.zip'
NGLD_EXTRACT_DIR = f'{DATA_CACHE_DIR}/ngld_raw'

if not os.path.exists(NGLD_ZIP):
    raise FileNotFoundError(
        'NGLD.zip not found in manual_downloads/. Follow the 4 steps in the markdown '
        'cell above, upload the zip to Drive, then re-run this cell.'
    )

if not os.path.exists(NGLD_EXTRACT_DIR):
    os.makedirs(NGLD_EXTRACT_DIR, exist_ok=True)
    !unzip -q -o "{NGLD_ZIP}" -d "{NGLD_EXTRACT_DIR}"
    print('Extracted NGLD.')
else:
    print('NGLD already extracted, skipping.')

def find_leaf_dirs(root):
    """Return {folder_name_lower: full_path} for every leaf directory under root
    that directly contains image files (handles unknown nesting inside the zip)."""
    found = {}
    for dirpath, dirnames, filenames in os.walk(root):
        imgs = [f for f in filenames if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        if imgs:
            found[os.path.basename(dirpath).lower()] = dirpath
    return found

ngld_dirs = find_leaf_dirs(NGLD_EXTRACT_DIR)
print('Found NGLD class folders:', list(ngld_dirs.keys()))


In [ ]:
# Map NGLD's folder names (verify against the printed list above and adjust if the
# actual zip uses different spelling/casing — dataset folder names on Mendeley are
# not always identical to the paper's prose) to our canonical grape classes.
NGLD_NAME_MAP = {
    'downy mildew': 'Grape___Downy_Mildew',
    'powdery mildew': 'Grape___Powdery_Mildew',
    'bacterial rot': 'Grape___Bacterial_Rot',
    'bacterial leaf spot': 'Grape___Bacterial_Rot',
    'healthy': 'Grape___healthy',
    'healthy leaves': 'Grape___healthy',
}

import random
random.seed(42)

NGLD_HOLDOUT_FRACTION = 0.15  # held out into field_eval, never trained on

n_merged = {'field_finetune': 0, 'field_eval': 0}
for folder_name, full_path in ngld_dirs.items():
    canon = NGLD_NAME_MAP.get(folder_name)
    if canon is None:
        print(f'  Skipping unmapped NGLD folder: {folder_name!r} (add it to NGLD_NAME_MAP if this should be included)')
        continue
    files = [f for f in os.listdir(full_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    random.shuffle(files)
    n_holdout = max(1, int(len(files) * NGLD_HOLDOUT_FRACTION))
    holdout_files, train_files = files[:n_holdout], files[n_holdout:]

    ft_dst = f'{DATA_CACHE_DIR}/grape/field_finetune/{canon}'
    ev_dst = f'{DATA_CACHE_DIR}/grape/field_eval/{canon}'
    os.makedirs(ft_dst, exist_ok=True)
    os.makedirs(ev_dst, exist_ok=True)

    for f in train_files:
        dst = os.path.join(ft_dst, f'ngld_{f}')
        if not os.path.exists(dst):
            shutil.copy2(os.path.join(full_path, f), dst)
            n_merged['field_finetune'] += 1
    for f in holdout_files:
        dst = os.path.join(ev_dst, f'ngld_{f}')
        if not os.path.exists(dst):
            shutil.copy2(os.path.join(full_path, f), dst)
            n_merged['field_eval'] += 1

print(f"Merged {n_merged['field_finetune']} NGLD images into grape field_finetune")
print(f"Merged {n_merged['field_eval']} NGLD images into grape field_eval (held out)")


## Section 6 — Bangladesh Tomato Leaf Dataset — tomato field-data top-up



In [ ]:
BD_ZIP = f'{MANUAL_DOWNLOADS_DIR}/BD_Tomato.zip'
BD_EXTRACT_DIR = f'{DATA_CACHE_DIR}/bd_tomato_raw'

if not os.path.exists(BD_ZIP):
    raise FileNotFoundError(
        'BD_Tomato.zip not found in manual_downloads/. Follow the steps in the markdown '
        'cell above, upload the zip to Drive, then re-run this cell.'
    )

if not os.path.exists(BD_EXTRACT_DIR):
    os.makedirs(BD_EXTRACT_DIR, exist_ok=True)
    !unzip -q -o "{BD_ZIP}" -d "{BD_EXTRACT_DIR}"
    print('Extracted Bangladesh tomato dataset.')
else:
    print('Already extracted, skipping.')

bd_dirs = find_leaf_dirs(BD_EXTRACT_DIR)
print('Found folders:', list(bd_dirs.keys()))


In [ ]:
BD_NAME_MAP = {
    'early blight': 'Tomato___Early_blight',
    'late blight': 'Tomato___Late_blight',
    'leaf mold': 'Tomato___Leaf_Mold',
    'bacterial spot': 'Tomato___Bacterial_spot',
    'target spot': 'Tomato___Target_Spot',
    'healthy': 'Tomato___healthy',
    # deliberately NOT mapped: 'black spot' has no equivalent PlantVillage/PlantDoc class
}

BD_HOLDOUT_FRACTION = 0.15

n_merged = {'field_finetune': 0, 'field_eval': 0}
for folder_name, full_path in bd_dirs.items():
    canon = BD_NAME_MAP.get(folder_name)
    if canon is None:
        print(f'  Excluding {folder_name!r} (no matching class in our taxonomy)')
        continue
    files = [f for f in os.listdir(full_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    random.shuffle(files)
    n_holdout = max(1, int(len(files) * BD_HOLDOUT_FRACTION))
    holdout_files, train_files = files[:n_holdout], files[n_holdout:]

    ft_dst = f'{DATA_CACHE_DIR}/tomato/field_finetune/{canon}'
    ev_dst = f'{DATA_CACHE_DIR}/tomato/field_eval/{canon}'
    os.makedirs(ft_dst, exist_ok=True)
    os.makedirs(ev_dst, exist_ok=True)

    for f in train_files:
        dst = os.path.join(ft_dst, f'bd_{f}')
        if not os.path.exists(dst):
            shutil.copy2(os.path.join(full_path, f), dst)
            n_merged['field_finetune'] += 1
    for f in holdout_files:
        dst = os.path.join(ev_dst, f'bd_{f}')
        if not os.path.exists(dst):
            shutil.copy2(os.path.join(full_path, f), dst)
            n_merged['field_eval'] += 1

print(f"Merged {n_merged['field_finetune']} Bangladesh images into tomato field_finetune")
print(f"Merged {n_merged['field_eval']} Bangladesh images into tomato field_eval (held out)")


## Section 7 — Dataset sanity check: see the actual images

Same as v8/v9 — a quick visual gut-check before spending GPU hours. This also lets
you visually confirm the NGLD/Bangladesh images look right in their assigned folders.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image as PILImage

def browse_random_images(crop, split, n=8):
    root = f'{DATA_CACHE_DIR}/{crop}/{split}'
    all_imgs = glob.glob(f'{root}/*/*')
    if not all_imgs:
        print(f'No images found under {root}')
        return
    sample = random.sample(all_imgs, min(n, len(all_imgs)))
    cols = 4
    rows = (len(sample) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
    axes = axes.flatten() if rows > 1 else [axes] if cols == 1 else axes
    for ax, path in zip(axes, sample):
        img = PILImage.open(path).convert('RGB')
        ax.imshow(img)
        cls = os.path.basename(os.path.dirname(path))
        ax.set_title(cls, fontsize=8)
        ax.axis('off')
    for ax in axes[len(sample):]:
        ax.axis('off')
    plt.suptitle(f'{crop} / {split}')
    plt.tight_layout()
    plt.show()

# Try these to eyeball each source:
browse_random_images('grape', 'field_finetune', n=8)
browse_random_images('tomato', 'field_finetune', n=8)


## Section 8 — Transforms, model builder, class weights

aspect-ratio-preserving
resize, moderate augmentation only, class-weighted loss for imbalance.

In [ ]:
import torch
import torch.nn as nn
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
import timm

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
IMG_SIZE = 224
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_tfm = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.CenterCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.RandomPerspective(distortion_scale=0.2, p=0.3),
    transforms.ColorJitter(0.15, 0.15, 0.15),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_tfm = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

def build_model(arch_name, num_classes):
    return timm.create_model(arch_name, pretrained=True, num_classes=num_classes)

print('Device:', DEVICE)


In [ ]:
class FixedClassFolderDataset(Dataset):
    """Maps images to a pre-defined class list (not whatever subset of classes
    happens to be present in a given folder) so index 3 always means the same
    disease across lab_train/lab_val/field_finetune/field_eval, even though
    smaller folders (field data) don't cover every class."""

    def __init__(self, root_dir, classes, transform):
        self.transform = transform
        self.classes = classes
        self.class_to_idx = {c: i for i, c in enumerate(classes)}
        self.samples = []
        for cls in classes:
            cls_dir = os.path.join(root_dir, cls)
            if not os.path.isdir(cls_dir):
                continue
            for fname in os.listdir(cls_dir):
                if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
                    self.samples.append((os.path.join(cls_dir, fname), self.class_to_idx[cls]))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = PILImage.open(path).convert('RGB')
        return self.transform(img), label


def make_class_weights(dataset, num_classes):
    counts = torch.zeros(num_classes)
    for _, label in dataset.samples:
        counts[label] += 1
    counts = torch.clamp(counts, min=1)
    weights = counts.sum() / (num_classes * counts)
    return weights.to(DEVICE)


## Section 9 — Training function (3-stage, checkpointed, resumable)


In [ ]:
def train_crop_model(crop, arch_name, epochs_head=3, epochs_finetune=5, epochs_field_finetune=12, base_lr=1e-4):
    classes = CROP_CLASSES[crop]
    num_classes = len(classes)

    train_ds = FixedClassFolderDataset(f'{DATA_CACHE_DIR}/{crop}/lab_train', classes, train_tfm)
    val_ds = FixedClassFolderDataset(f'{DATA_CACHE_DIR}/{crop}/lab_val', classes, eval_tfm)
    field_ds = FixedClassFolderDataset(f'{DATA_CACHE_DIR}/{crop}/field_finetune', classes, train_tfm)

    train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=2)
    field_loader = DataLoader(field_ds, batch_size=16, shuffle=True, num_workers=2) if len(field_ds) > 0 else None

    class_weights = make_class_weights(train_ds, num_classes)
    criterion = nn.CrossEntropyLoss(weight=class_weights)

    model = build_model(arch_name, num_classes).to(DEVICE)

    ckpt_path = f'{CKPT_DIR}/{crop}_{arch_name}.pt'
    start_epoch = 0
    if os.path.exists(ckpt_path):
        ckpt = torch.load(ckpt_path, map_location=DEVICE)
        if ckpt.get('arch_name') == arch_name and ckpt.get('crop') == crop and ckpt.get('classes') == classes:
            model.load_state_dict(ckpt['model_state'])
            start_epoch = ckpt['epoch'] + 1
            print(f'Resuming {crop}/{arch_name} from epoch {start_epoch}')
        else:
            print(f'Checkpoint at {ckpt_path} does not match current config (arch/crop/classes) — starting fresh.')

    total_epochs = epochs_head + epochs_finetune + epochs_field_finetune
    stage_bounds = {
        'head': (0, epochs_head),
        'finetune': (epochs_head, epochs_head + epochs_finetune),
        'field_finetune': (epochs_head + epochs_finetune, total_epochs),
    }

    def stage_for_epoch(e):
        for name, (lo, hi) in stage_bounds.items():
            if lo <= e < hi:
                return name
        return 'field_finetune'

    prev_stage, opt, scheduler = None, None, None

    for epoch in range(start_epoch, total_epochs):
        stage = stage_for_epoch(epoch)

        if stage != prev_stage:
            if stage == 'head':
                for p in model.parameters():
                    p.requires_grad = False
                for p in model.get_classifier().parameters():
                    p.requires_grad = True
                stage_epochs = epochs_head
                loader = train_loader
            elif stage == 'finetune':
                for p in model.parameters():
                    p.requires_grad = True
                stage_epochs = epochs_finetune
                loader = train_loader
            else:  # field_finetune — backbone frozen again to avoid catastrophic forgetting
                for p in model.parameters():
                    p.requires_grad = False
                for p in model.get_classifier().parameters():
                    p.requires_grad = True
                stage_epochs = epochs_field_finetune
                loader = field_loader if field_loader is not None else train_loader

            params = [p for p in model.parameters() if p.requires_grad]
            opt = torch.optim.Adam(params, lr=base_lr)
            scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(stage_epochs, 1))
            prev_stage = stage

        model.train()
        running_loss, running_correct, n = 0.0, 0, 0
        for imgs, labels in loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            opt.zero_grad()
            out = model(imgs)
            loss = criterion(out, labels)
            loss.backward()
            opt.step()
            running_loss += loss.item() * imgs.size(0)
            running_correct += (out.argmax(1) == labels).sum().item()
            n += imgs.size(0)
        scheduler.step()

        model.eval()
        val_correct, val_n = 0, 0
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
                out = model(imgs)
                val_correct += (out.argmax(1) == labels).sum().item()
                val_n += imgs.size(0)

        train_acc = running_correct / max(n, 1)
        val_acc = val_correct / max(val_n, 1)
        print(f'[{crop}/{arch_name}] epoch {epoch+1}/{total_epochs} stage={stage} '
              f'train_loss={running_loss/max(n,1):.4f} train_acc={train_acc:.4f} lab_val_acc={val_acc:.4f}')

        torch.save({
            'model_state': model.state_dict(),
            'epoch': epoch,
            'arch_name': arch_name,
            'crop': crop,
            'classes': classes,
        }, ckpt_path)

    torch.save(model.state_dict(), f'{MODELS_DIR}/{crop}_{arch_name}_final.pt')
    return model


## Section 10 — Train tomato (ensemble: MobileNetV3 + ResNet50)

Per-crop config carried over from v9's evidence table: tomato does best as an
ensemble of these two architectures, no TTA.

In [ ]:
tomato_mobilenet = train_crop_model('tomato', 'mobilenetv3_large_100')


In [ ]:
tomato_resnet = train_crop_model('tomato', 'resnet50')


## Section 11 — Train grape (single MobileNetV3, no TTA)


In [ ]:
grape_mobilenet = train_crop_model('grape', 'mobilenetv3_large_100')


## Section 12 — Evaluation helpers (single model, ensemble, TTA)

In [ ]:
import torch.nn.functional as F
from sklearn.metrics import classification_report, accuracy_score

def load_final(crop, arch_name):
    classes = CROP_CLASSES[crop]
    model = build_model(arch_name, len(classes)).to(DEVICE)
    model.load_state_dict(torch.load(f'{MODELS_DIR}/{crop}_{arch_name}_final.pt', map_location=DEVICE))
    model.eval()
    return model

def predict_logits(models, imgs, tta=False):
    with torch.no_grad():
        probs_sum = 0
        for model in models:
            probs_sum = probs_sum + F.softmax(model(imgs), dim=1)
            if tta:
                probs_sum = probs_sum + F.softmax(model(torch.flip(imgs, dims=[3])), dim=1)
        return probs_sum / (len(models) * (2 if tta else 1))

def evaluate_variant(crop, models, tta=False, eval_split='field_eval'):
    classes = CROP_CLASSES[crop]
    ds = FixedClassFolderDataset(f'{DATA_CACHE_DIR}/{crop}/{eval_split}', classes, eval_tfm)
    if len(ds) == 0:
        print(f'No images in {crop}/{eval_split}, skipping.')
        return None
    loader = DataLoader(ds, batch_size=32, shuffle=False, num_workers=2)
    all_preds, all_labels = [], []
    for imgs, labels in loader:
        imgs = imgs.to(DEVICE)
        probs = predict_logits(models, imgs, tta=tta)
        all_preds.extend(probs.argmax(1).cpu().tolist())
        all_labels.extend(labels.tolist())
    acc = accuracy_score(all_labels, all_preds)
    print(classification_report(all_labels, all_preds, labels=list(range(len(classes))), target_names=classes, zero_division=0))
    return acc


## Section 13 — Evidence table: which config actually wins on the combined field test set?

Same evidence-based comparison approach as v9 (this is what caught the ensemble/TTA
mistake last time)

In [ ]:
tomato_models_all = [load_final('tomato', 'mobilenetv3_large_100'), load_final('tomato', 'resnet50')]
grape_models_all = [load_final('grape', 'mobilenetv3_large_100')]

results = []
print('=== TOMATO ===')
print('-- mobilenet alone, no TTA --')
results.append(('tomato', 'mobilenet', False, evaluate_variant('tomato', [tomato_models_all[0]], tta=False)))
print('-- resnet alone, no TTA --')
results.append(('tomato', 'resnet', False, evaluate_variant('tomato', [tomato_models_all[1]], tta=False)))
print('-- ensemble, no TTA --')
results.append(('tomato', 'ensemble', False, evaluate_variant('tomato', tomato_models_all, tta=False)))
print('-- ensemble, with TTA --')
results.append(('tomato', 'ensemble', True, evaluate_variant('tomato', tomato_models_all, tta=True)))

print('\n=== GRAPE ===')
print('-- mobilenet alone, no TTA --')
results.append(('grape', 'mobilenet', False, evaluate_variant('grape', grape_models_all, tta=False)))
print('-- mobilenet alone, with TTA --')
results.append(('grape', 'mobilenet', True, evaluate_variant('grape', grape_models_all, tta=True)))

print('\n=== SUMMARY ===')
for crop, variant, tta, acc in results:
    print(f'{crop:8s} {variant:10s} TTA={tta!s:5s} acc={acc:.4f}' if acc is not None else f'{crop} {variant} TTA={tta} -- no data --')


## Section 14 — Live single-image test

Pick whichever config Section 13 showed as the winner for each crop before
demoing to judges — update `TOMATO_BEST_MODE` / `GRAPE_BEST_MODE` below to match.

In [ ]:
TOMATO_BEST_MODE = 'ensemble'   # 'mobilenet' | 'resnet' | 'ensemble' -- set from Section 13's winner
TOMATO_BEST_TTA = False
GRAPE_BEST_MODE = 'mobilenet'
GRAPE_BEST_TTA = False

def models_for(crop, mode):
    if crop == 'tomato':
        return {'mobilenet': [tomato_models_all[0]], 'resnet': [tomato_models_all[1]], 'ensemble': tomato_models_all}[mode]
    return {'mobilenet': grape_models_all}[mode]

def predict(image_path, crop):
    classes = CROP_CLASSES[crop]
    mode, tta = (TOMATO_BEST_MODE, TOMATO_BEST_TTA) if crop == 'tomato' else (GRAPE_BEST_MODE, GRAPE_BEST_TTA)
    models = models_for(crop, mode)
    img = PILImage.open(image_path).convert('RGB')
    x = eval_tfm(img).unsqueeze(0).to(DEVICE)
    probs = predict_logits(models, x, tta=tta)[0]
    top_idx = probs.argmax().item()
    plt.imshow(img)
    plt.axis('off')
    plt.title(f'{classes[top_idx]}  ({probs[top_idx]*100:.1f}% confidence)')
    plt.show()
    top5 = probs.topk(min(5, len(classes)))
    for idx, p in zip(top5.indices.tolist(), top5.values.tolist()):
        print(f'  {classes[idx]:45s} {p*100:5.1f}%')

from google.colab import files
print('Upload a leaf photo (tomato or grape):')
uploaded = files.upload()
uploaded_path = list(uploaded.keys())[0]
predict(uploaded_path, crop='tomato')  # change to crop='grape' as needed


## Section 15 — Export metadata (for the demo notebook / app integration)

In [ ]:
import json as _json

meta = {
    'tomato_classes': TOMATO_CLASSES,
    'grape_classes': GRAPE_CLASSES,
    'tomato_best_mode': TOMATO_BEST_MODE,
    'tomato_best_tta': TOMATO_BEST_TTA,
    'grape_best_mode': GRAPE_BEST_MODE,
    'grape_best_tta': GRAPE_BEST_TTA,
    'field_eval_results': [
        {'crop': c, 'variant': v, 'tta': t, 'accuracy': a} for c, v, t, a in results if a is not None
    ],
}

with open(f'{MODELS_DIR}/meta.json', 'w') as f:
    _json.dump(meta, f, indent=2)

print('Saved meta.json to', MODELS_DIR)
print(_json.dumps(meta, indent=2))
